# Introducción a Mexican Insurance Analytics Suite

## Productos de Vida: Temporal, Ordinario y Dotal

Este notebook demuestra el uso de la librería `mexican_insurance` para calcular primas y reservas de productos de seguros de vida.

### Contenido
1. Instalación y configuración
2. Carga de tabla de mortalidad EMSSA-09
3. Seguro de Vida Temporal
4. Seguro de Vida Ordinario (Vitalicio)
5. Seguro de Vida Dotal (Mixto)
6. Comparativa entre productos
7. Visualizaciones

## 1. Instalación y Configuración

Si aún no has instalado la librería, ejecuta:

```bash
pip install -e ".[viz]"
```

In [ ]:
# Imports necesarios
from decimal import Decimal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports de la librería
from mexican_insurance.actuarial.mortality.tablas import TablaMortalidad
from mexican_insurance.products.vida.temporal import VidaTemporal
from mexican_insurance.products.vida.ordinario import VidaOrdinario
from mexican_insurance.products.vida.dotal import VidaDotal
from mexican_insurance.core.validators import (
    Asegurado,
    ConfiguracionProducto,
    Sexo,
    Fumador
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports completados exitosamente")

## 2. Carga de Tabla de Mortalidad EMSSA-09

La tabla EMSSA-09 es una de las tablas de mortalidad más utilizadas en México para seguros de vida.

In [ ]:
# Cargar tabla de mortalidad EMSSA-09
tabla = TablaMortalidad.cargar_emssa09()

print(f"Tabla cargada: {tabla.nombre}")
print(f"Sexo: {tabla.sexo}")
print(f"Edad mínima: {tabla.edad_minima}")
print(f"Edad máxima: {tabla.edad_maxima}")
print(f"Total edades: {len(tabla.qx)} registros")

# Ver primeras probabilidades de muerte
print("\n📊 Primeras 10 probabilidades de muerte (qx):")
for edad in range(25, 35):
    qx = tabla.obtener_qx(edad)
    print(f"  Edad {edad}: qx = {qx:.6f} ({qx*1000:.3f} por mil)")

## 3. Seguro de Vida Temporal

El seguro temporal proporciona cobertura por un plazo fijo. Si el asegurado fallece durante el plazo, se paga la suma asegurada. Si sobrevive, no hay pago (seguro de riesgo puro).

In [ ]:
# Configurar producto: Vida Temporal 20 años
config_temporal = ConfiguracionProducto(
    nombre_producto="Vida Temporal 20 años",
    plazo_years=20,
    tasa_interes_tecnico=Decimal("0.055"),  # 5.5% anual
    gastos_adquisicion=Decimal("0.05"),     # 5% gastos de adquisición
    gastos_administracion=Decimal("0.02"),  # 2% gastos de administración
    margen_utilidad=Decimal("0.10")         # 10% margen de utilidad
)

# Crear producto
producto_temporal = VidaTemporal(config_temporal, tabla)

print(f"✅ Producto creado: {config_temporal.nombre_producto}")
print(f"   Plazo: {config_temporal.plazo_years} años")
print(f"   Tasa técnica: {config_temporal.tasa_interes_tecnico*100}%")

In [ ]:
# Definir asegurado
asegurado_1 = Asegurado(
    edad=35,
    sexo=Sexo.HOMBRE,
    suma_asegurada=Decimal("1000000")  # 1 millón de pesos
)

# Calcular prima
resultado_temporal = producto_temporal.calcular_prima(
    asegurado_1, 
    frecuencia_pago="mensual"
)

print(f"\n💰 RESULTADO VIDA TEMPORAL")
print(f"="*50)
print(f"Asegurado: {asegurado_1.sexo.value}, {asegurado_1.edad} años")
print(f"Suma Asegurada: ${asegurado_1.suma_asegurada:,.2f} MXN")
print(f"\nPrima Neta Mensual: ${resultado_temporal.prima_neta:,.2f} MXN")
print(f"Prima Total Mensual: ${resultado_temporal.prima_total:,.2f} MXN")
print(f"\nDesglose de Recargos:")
for concepto, monto in resultado_temporal.desglose_recargos.items():
    print(f"  - {concepto}: ${monto:,.2f} MXN")

In [ ]:
# Calcular reserva a diferentes edades
print("\n📈 EVOLUCIÓN DE RESERVA MATEMÁTICA")
print("="*50)

edades_reserva = [35, 40, 45, 50, 54]  # Hasta antes de terminar el plazo (55)
reservas_temporal = []

for edad_actual in edades_reserva:
    asegurado_temp = Asegurado(
        edad=edad_actual,
        sexo=Sexo.HOMBRE,
        suma_asegurada=Decimal("1000000")
    )
    tiempo_transcurrido = edad_actual - 35
    reserva = producto_temporal.calcular_reserva(
        asegurado_temp,
        tiempo_transcurrido=tiempo_transcurrido
    )
    reservas_temporal.append(float(reserva))
    print(f"Año {tiempo_transcurrido} (edad {edad_actual}): ${reserva:,.2f} MXN")

## 4. Seguro de Vida Ordinario (Vitalicio)

El seguro ordinario proporciona cobertura vitalicia. Se paga la suma asegurada cuando fallezca el asegurado, sin importar cuándo ocurra.

In [ ]:
# Configurar producto: Vida Ordinario (pago hasta 65 años)
config_ordinario = ConfiguracionProducto(
    nombre_producto="Vida Ordinario con pago hasta 65 años",
    plazo_years=99,  # Cobertura vitalicia
    plazo_pago_years=30,  # Pago limitado hasta edad 65 (35+30)
    tasa_interes_tecnico=Decimal("0.050"),
    gastos_adquisicion=Decimal("0.05"),
    gastos_administracion=Decimal("0.02"),
    margen_utilidad=Decimal("0.10")
)

producto_ordinario = VidaOrdinario(config_ordinario, tabla)

# Calcular prima para el mismo asegurado
resultado_ordinario = producto_ordinario.calcular_prima(
    asegurado_1,
    frecuencia_pago="mensual"
)

print(f"💰 RESULTADO VIDA ORDINARIO")
print(f"="*50)
print(f"Asegurado: {asegurado_1.sexo.value}, {asegurado_1.edad} años")
print(f"Suma Asegurada: ${asegurado_1.suma_asegurada:,.2f} MXN")
print(f"Plazo de pago: {config_ordinario.plazo_pago_years} años (hasta edad 65)")
print(f"\nPrima Neta Mensual: ${resultado_ordinario.prima_neta:,.2f} MXN")
print(f"Prima Total Mensual: ${resultado_ordinario.prima_total:,.2f} MXN")
print(f"\nDesglose de Recargos:")
for concepto, monto in resultado_ordinario.desglose_recargos.items():
    print(f"  - {concepto}: ${monto:,.2f} MXN")

In [ ]:
# Calcular reservas para Vida Ordinario
print("\n📈 EVOLUCIÓN DE RESERVA MATEMÁTICA (ORDINARIO)")
print("="*50)

reservas_ordinario = []
for edad_actual in edades_reserva:
    asegurado_temp = Asegurado(
        edad=edad_actual,
        sexo=Sexo.HOMBRE,
        suma_asegurada=Decimal("1000000")
    )
    tiempo_transcurrido = edad_actual - 35
    reserva = producto_ordinario.calcular_reserva(
        asegurado_temp,
        tiempo_transcurrido=tiempo_transcurrido
    )
    reservas_ordinario.append(float(reserva))
    print(f"Año {tiempo_transcurrido} (edad {edad_actual}): ${reserva:,.2f} MXN")

## 5. Seguro de Vida Dotal (Mixto)

El seguro dotal combina protección y ahorro. Paga la suma asegurada si el asegurado fallece durante el plazo, O si sobrevive al final del plazo.

In [ ]:
# Configurar producto: Vida Dotal 20 años
config_dotal = ConfiguracionProducto(
    nombre_producto="Vida Dotal 20 años",
    plazo_years=20,
    tasa_interes_tecnico=Decimal("0.045"),  # 4.5% (menor por el ahorro)
    gastos_adquisicion=Decimal("0.05"),
    gastos_administracion=Decimal("0.02"),
    margen_utilidad=Decimal("0.10")
)

producto_dotal = VidaDotal(config_dotal, tabla)

# Calcular prima
resultado_dotal = producto_dotal.calcular_prima(
    asegurado_1,
    frecuencia_pago="mensual"
)

print(f"💰 RESULTADO VIDA DOTAL")
print(f"="*50)
print(f"Asegurado: {asegurado_1.sexo.value}, {asegurado_1.edad} años")
print(f"Suma Asegurada: ${asegurado_1.suma_asegurada:,.2f} MXN")
print(f"\nPrima Neta Mensual: ${resultado_dotal.prima_neta:,.2f} MXN")
print(f"Prima Total Mensual: ${resultado_dotal.prima_total:,.2f} MXN")
print(f"\nDesglose de Recargos:")
for concepto, monto in resultado_dotal.desglose_recargos.items():
    print(f"  - {concepto}: ${monto:,.2f} MXN")

print(f"\n🎯 Nota: Prima más alta porque incluye componente de ahorro")

In [ ]:
# Calcular reservas para Vida Dotal
print("\n📈 EVOLUCIÓN DE RESERVA MATEMÁTICA (DOTAL)")
print("="*50)

reservas_dotal = []
for edad_actual in edades_reserva:
    asegurado_temp = Asegurado(
        edad=edad_actual,
        sexo=Sexo.HOMBRE,
        suma_asegurada=Decimal("1000000")
    )
    tiempo_transcurrido = edad_actual - 35
    reserva = producto_dotal.calcular_reserva(
        asegurado_temp,
        tiempo_transcurrido=tiempo_transcurrido
    )
    reservas_dotal.append(float(reserva))
    print(f"Año {tiempo_transcurrido} (edad {edad_actual}): ${reserva:,.2f} MXN")

## 6. Comparativa entre Productos

Comparemos las primas y reservas de los tres productos para el mismo asegurado.

In [ ]:
# Tabla comparativa de primas
comparativa_df = pd.DataFrame({
    'Producto': ['Vida Temporal', 'Vida Ordinario', 'Vida Dotal'],
    'Prima Neta': [
        float(resultado_temporal.prima_neta),
        float(resultado_ordinario.prima_neta),
        float(resultado_dotal.prima_neta)
    ],
    'Prima Total': [
        float(resultado_temporal.prima_total),
        float(resultado_ordinario.prima_total),
        float(resultado_dotal.prima_total)
    ],
    'Tipo Cobertura': [
        'Solo fallecimiento',
        'Fallecimiento vitalicio',
        'Fallecimiento + Supervivencia'
    ]
})

print("\n📊 COMPARATIVA DE PRODUCTOS")
print("="*70)
print(comparativa_df.to_string(index=False))

# Calcular diferencias porcentuales
print("\n💡 Análisis:")
prima_temp = float(resultado_temporal.prima_total)
prima_ord = float(resultado_ordinario.prima_total)
prima_dot = float(resultado_dotal.prima_total)

diff_ord = ((prima_ord - prima_temp) / prima_temp) * 100
diff_dot = ((prima_dot - prima_temp) / prima_temp) * 100

print(f"  - Vida Ordinario es {diff_ord:.1f}% más cara que Temporal")
print(f"  - Vida Dotal es {diff_dot:.1f}% más cara que Temporal")
print(f"  - Dotal tiene el mayor ahorro acumulado al final del plazo")

## 7. Visualizaciones

Grafiquemos la evolución de reservas y comparemos los productos visualmente.

In [ ]:
# Gráfica 1: Comparación de Primas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Primas mensuales
productos = ['Temporal', 'Ordinario', 'Dotal']
primas_netas = [prima_temp, prima_ord, prima_dot]
colores = ['#3498db', '#e74c3c', '#2ecc71']

bars = ax1.bar(productos, primas_netas, color=colores, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Prima Mensual (MXN)', fontsize=12)
ax1.set_title('Comparación de Primas Mensuales\nAsegurado: Hombre, 35 años, SA: $1,000,000', 
              fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Agregar valores sobre las barras
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:,.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Gráfica 2: Evolución de Reservas
años = [0, 5, 10, 15, 19]
ax2.plot(años, reservas_temporal, marker='o', linewidth=2.5, 
         label='Temporal', color=colores[0], markersize=8)
ax2.plot(años, reservas_ordinario, marker='s', linewidth=2.5, 
         label='Ordinario', color=colores[1], markersize=8)
ax2.plot(años, reservas_dotal, marker='^', linewidth=2.5, 
         label='Dotal', color=colores[2], markersize=8)

ax2.set_xlabel('Años desde emisión', fontsize=12)
ax2.set_ylabel('Reserva Matemática (MXN)', fontsize=12)
ax2.set_title('Evolución de Reservas Matemáticas', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

print("\n✅ Gráficas generadas exitosamente")

In [ ]:
# Gráfica 3: Análisis de sensibilidad por edad
fig, ax = plt.subplots(figsize=(12, 6))

edades = range(25, 56, 5)  # De 25 a 55 años cada 5 años
primas_temp_edad = []
primas_ord_edad = []
primas_dot_edad = []

for edad in edades:
    aseg = Asegurado(edad=edad, sexo=Sexo.HOMBRE, suma_asegurada=Decimal("1000000"))
    
    # Temporal
    res_temp = producto_temporal.calcular_prima(aseg, frecuencia_pago="mensual")
    primas_temp_edad.append(float(res_temp.prima_total))
    
    # Ordinario
    res_ord = producto_ordinario.calcular_prima(aseg, frecuencia_pago="mensual")
    primas_ord_edad.append(float(res_ord.prima_total))
    
    # Dotal
    res_dot = producto_dotal.calcular_prima(aseg, frecuencia_pago="mensual")
    primas_dot_edad.append(float(res_dot.prima_total))

ax.plot(edades, primas_temp_edad, marker='o', linewidth=2.5, 
        label='Temporal 20 años', color=colores[0], markersize=8)
ax.plot(edades, primas_ord_edad, marker='s', linewidth=2.5, 
        label='Ordinario (pago hasta 65)', color=colores[1], markersize=8)
ax.plot(edades, primas_dot_edad, marker='^', linewidth=2.5, 
        label='Dotal 20 años', color=colores[2], markersize=8)

ax.set_xlabel('Edad de Entrada', fontsize=12)
ax.set_ylabel('Prima Mensual (MXN)', fontsize=12)
ax.set_title('Sensibilidad de Primas según Edad de Entrada\nSuma Asegurada: $1,000,000 MXN', 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

print("\n📊 Observaciones:")
print("  - Las primas aumentan con la edad debido a mayor mortalidad")
print("  - Dotal siempre es más caro por incluir componente de ahorro")
print("  - Ordinario es más caro que Temporal por cobertura vitalicia")

## 8. Análisis de Cartera de Ejemplo

Carguemos la cartera de ejemplo y analicemos las primas totales.

In [ ]:
# Cargar datos de cartera
cartera_df = pd.read_csv('../examples/data/cartera_ejemplo.csv')

print("📂 CARTERA DE EJEMPLO")
print("="*70)
print(f"Total de pólizas: {len(cartera_df)}")
print(f"\nDistribución por producto:")
print(cartera_df['producto'].value_counts())
print(f"\nSuma asegurada total: ${cartera_df['suma_asegurada'].sum():,.2f} MXN")
print(f"Prima mensual total: ${cartera_df['prima_mensual'].sum():,.2f} MXN")
print(f"Prima anual total: ${cartera_df['prima_mensual'].sum() * 12:,.2f} MXN")

print("\n📋 Primeras 5 pólizas:")
print(cartera_df.head().to_string(index=False))

In [ ]:
# Análisis por producto
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribución de productos
producto_counts = cartera_df['producto'].value_counts()
axes[0, 0].pie(producto_counts.values, labels=producto_counts.index, 
               autopct='%1.1f%%', startangle=90, colors=colores)
axes[0, 0].set_title('Distribución de Productos', fontsize=12, fontweight='bold')

# Distribución por sexo
sexo_counts = cartera_df['sexo'].value_counts()
axes[0, 1].bar(sexo_counts.index, sexo_counts.values, color=['#3498db', '#e74c3c'], alpha=0.7)
axes[0, 1].set_title('Distribución por Sexo', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Cantidad de Pólizas')

# Distribución de edades
axes[1, 0].hist(cartera_df['edad'], bins=15, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Distribución de Edades', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Edad')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].axvline(cartera_df['edad'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f'Media: {cartera_df["edad"].mean():.1f}')
axes[1, 0].legend()

# Suma asegurada por producto
suma_por_producto = cartera_df.groupby('producto')['suma_asegurada'].sum() / 1000000
axes[1, 1].barh(suma_por_producto.index, suma_por_producto.values, color=colores, alpha=0.7)
axes[1, 1].set_title('Suma Asegurada Total por Producto', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Suma Asegurada (Millones MXN)')

plt.tight_layout()
plt.show()

print("\n✅ Análisis de cartera completado")

## Conclusiones

En este notebook aprendimos:

1. Cómo cargar y usar tablas de mortalidad EMSSA-09
2. Calcular primas para tres tipos de seguros de vida:
   - **Temporal**: Protección pura por plazo fijo
   - **Ordinario**: Cobertura vitalicia con pago limitado
   - **Dotal**: Combinación de protección y ahorro
3. Calcular reservas matemáticas y su evolución
4. Comparar productos y analizar sensibilidad por edad
5. Analizar una cartera de pólizas

### Próximos Pasos

Continúa con los siguientes notebooks:
- **02_reaseguro.ipynb**: Estrategias de reaseguro
- **03_reservas_tecnicas.ipynb**: Métodos Chain Ladder, B-F, Bootstrap
- **04_cumplimiento_cnsf.ipynb**: Cálculo de RCS y cumplimiento regulatorio
- **05_validaciones_sat.ipynb**: Validaciones fiscales
- **06_reportes_cnsf.ipynb**: Reportes automatizados
- **07_casos_practicos_completos.ipynb**: Workflows end-to-end